LEVEL 1 – 1단계: 자료형 심화 (실무 기준)

단순 문법이 아니라 운영 코드에서 사고 나는 지점 중심으로 간다.

1️⃣ Mutable vs Immutable — 왜 중요한가

| 구분        | 타입                     | 특징               |
| --------- | ---------------------- | ---------------- |
| Immutable | int, float, str, tuple | 값 변경 시 새로운 객체 생성 |
| Mutable   | list, dict, set        | 객체 내부 값 변경 가능    |


In [1]:
# 실무 장애 포인트
a = [1, 2, 3]
b = a
b.append(4)

print(a)  # ?

[1, 2, 3, 4]


*왜 문제인가?*

- 함수에 전달된 리스트가 원본을 오염

- 전역 변수 수정

- 캐시 데이터 파괴

- 멀티스레드 환경에서 예측 불가 동작

In [ ]:
# 실무 안전패턴
b = a.copy()      # 얕은 복사
b = list(a)       # 동일

# 중첩 구조라면
import copy
b = copy.deepcopy(a)

2️⃣ 얕은 복사 vs 깊은 복사

In [6]:
# 얕은 복사 예시
a = [[1,2], [3,4]]
b = a.copy()

b[0][0] = 999
print(a)
print(b)
# copy()는 1단계만 복사
# 내부리스트는 같은 참조

# 깊은 복사 예시
import copy
a = [[1,2], [3,4]]
b = copy.deepcopy(a)
b[0][0] = 999
print(a)
print(b)
# 중첩 구조에서도 모두 복사
# 메모리 소모 커짐


[[999, 2], [3, 4]]
[[999, 2], [3, 4]]
[[1, 2], [3, 4]]
[[999, 2], [3, 4]]


실무 판단 기준
| 상황        | 선택                     |
| --------- | ---------------------- |
| 설정값 복사    | deepcopy               |
| 단순 1차 리스트 | copy                   |
| 대용량 데이터   | deepcopy 주의 (메모리 비용 큼) |


3️⃣ dict 안전 접근 패턴 (실무 핵심)

In [8]:
# 위험한 코드
data = {}
value = data["result"]["items"][0]["name"]

# keyError / IndexError 발생 가능

KeyError: 'result'

In [ ]:
# 안전패턴
value = (
    data.get("result", {})
        .get("items", [{}])[0]
        .get("name")
)

# 또는 명확하게
try:
    value = data["result"]["items"][0]["name"]
except (KeyError, IndexError, TypeError):
    value = None

*설계관점*
- api응답은 항상 깨진다고 가정
- 외부 입력은 신뢰하지 않는다
- "fail-safe" 구조로 작성

4️⃣ 리스트/딕셔너리 컴프리헨션